In [ ]:
# code1_gnn_plus_kge.py
# 双支路：TransE + GraphSAGE → 拼接 → TransH

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import GraphSAINTNodeSampler, GraphSAINTRandomWalkSampler
# from torch_geometric.utils import negative_sampling
import random
import numpy as np
import os
from tqdm import tqdm
import faiss
import zipfile

# ==================== 固定随机种子 ====================
torch.backends.cudnn.deterministic = True
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# ==================== 配置 ====================
scheme_type = "s9_gnn_plus_kge"
BASE_DIR = "/mnt/d/forCoding_data/Tianchi_EcommerceKG"
TRAIN_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_train.tsv"
DEV_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_dev.tsv"
TEST_FILE_PATH = f"{BASE_DIR}/originalData/OpenBG500/OpenBG500_test.tsv"
OUTPUT_FILE_PATH = f"{BASE_DIR}/preprocessedData/OpenBG500_test.tsv"
MODEL_DIR = f"{BASE_DIR}/trained_models/{scheme_type}"
os.makedirs(MODEL_DIR, exist_ok=True)

TRAINED_MODEL_PATHS = {
    'TransE': f"{MODEL_DIR}/transE.pth",
    'GraphSAGE': f"{MODEL_DIR}/graphsage.pth",
    'TransH': f"{MODEL_DIR}/transH.pth"
}

# 超参数
EMBEDDING_DIM = 100
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5
EPOCHS = 1
BATCH_SIZE = 256
NEGATIVE_SAMPLES = 10
MAX_LINES = None
FORCE_RETRAIN = True

# ==================== 数据集 ====================
class KnowledgeGraphDataset(torch.utils.data.Dataset):
    def __init__(self, file_path, is_test=False, max_lines=None, is_train=False):
        self.triples = []
        self.is_train = is_train
        self._load_data(file_path, is_test, max_lines)
    def _load_data(self, file_path, is_test, max_lines):
        print(f"加载数据: {file_path}")
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if max_lines: lines = lines[:max_lines]
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 3:
                    h, r, t = parts
                    self.triples.append((h, r, t))
                elif is_test and len(parts) == 2:
                    h, r = parts
                    self.triples.append((h, r, "<UNK>"))
        print(f"共加载 {len(self.triples)} 个三元组")
    def __len__(self): return len(self.triples)
    def __getitem__(self, idx): return self.triples[idx]

def collate_fn(batch):
    h_list, r_list, t_list = zip(*batch)
    return list(h_list), list(r_list), list(t_list)

# ==================== 映射器 ====================
class EntityRelationMapper:
    def __init__(self):
        self.entity_to_id = {}
        self.id_to_entity = {}
        self.relation_to_id = {}
        self.id_to_relation = {}
        self.entity_count = 0
        self.relation_count = 0
        self.all_train_triples = []
    def build_mappings(self, *datasets):
        entities = set()
        relations = set()
        for dataset in datasets:
            for h, r, t in dataset.triples:
                entities.add(h); entities.add(t); relations.add(r)
                if dataset.is_train: self.all_train_triples.append((h, r, t))
        for e in sorted(entities):
            self.entity_to_id[e] = self.entity_count
            self.id_to_entity[self.entity_count] = e
            self.entity_count += 1
        for r in sorted(relations):
            self.relation_to_id[r] = self.relation_count
            self.id_to_relation[self.relation_count] = r
            self.relation_count += 1

# ==================== TransE ====================
class TransE(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)
    def forward(self, h, r, t):
        return torch.norm(self.E(h) + self.R(r) - self.E(t), p=1, dim=1)
    def get_query_embedding(self, h, r):
        return self.E(h) + self.R(r)
    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== 自定义负采样（修复 numpy 兼容性问题）====================
def custom_negative_sampling(edge_index, num_nodes, num_neg_samples=None):
    pos_edges = edge_index.cpu().numpy()
    pos_set = set(map(tuple, pos_edges.T))
    num_neg = num_neg_samples if num_neg_samples else edge_index.size(1)
    neg_edges = []
    for _ in range(num_neg):
        while True:
            h = random.randint(0, num_nodes - 1)
            t = random.randint(0, num_nodes - 1)
            if (h, t) not in pos_set and h != t:
                neg_edges.append([h, t])
                break
    return torch.tensor(neg_edges).t().contiguous().to(edge_index.device)

# ==================== GraphSAGE ====================
class GraphSAGE(nn.Module):
    def __init__(self, num_entities, dim):
        super().__init__()
        self.embedding = nn.Embedding(num_entities, dim)
        nn.init.xavier_uniform_(self.embedding.weight)
        self.conv1 = SAGEConv(dim, dim, aggr='mean')
        self.conv2 = SAGEConv(dim, dim, aggr='mean')
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = self.embedding(x)
        x = self.dropout(torch.relu(self.conv1(x, edge_index)))
        x = self.conv2(x, edge_index)
        return x

    def get_embedding_matrix(self, device):
        # ❌ 错误：不要用 with torch.no_grad()
        # ✅ 正确：只在推理时用 no_grad，在训练时直接 forward
        x = torch.arange(self.embedding.num_embeddings, device=device)
        return self.forward(x, self.edge_index)

# ==================== TransH ====================
class TransH(nn.Module):
    def __init__(self, num_entities, num_relations, dim):
        super().__init__()
        self.E = nn.Embedding(num_entities, dim)
        self.R = nn.Embedding(num_relations, dim)
        self.W = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.E.weight)
        nn.init.xavier_uniform_(self.R.weight)
        nn.init.xavier_uniform_(self.W.weight)
    def project(self, emb, w):
        norm_w = torch.nn.functional.normalize(w, p=2, dim=1)
        scale = torch.sum(emb * norm_w, dim=1, keepdim=True)
        return emb - scale * norm_w
    def forward(self, h, r, t):
        h_emb = self.E(h); t_emb = self.E(t); r_vec = self.R(r); W = self.W(r)
        h_proj = self.project(h_emb, W); t_proj = self.project(t_emb, W)
        return torch.norm(h_proj + r_vec - t_proj, p=1, dim=1)
    def get_query_embedding(self, h, r):
        h_emb = self.E(h); r_vec = self.R(r); W = self.W(r)
        h_proj = self.project(h_emb, W)
        return h_proj + r_vec
    def normalize_entities(self):
        with torch.no_grad():
            self.E.weight.data.div_(torch.norm(self.E.weight.data, dim=1, keepdim=True) + 1e-9)

# ==================== Train TransE ====================
def train_TransE(train_dataset, mapper, device):
    if os.path.exists(TRAINED_MODEL_PATHS['TransE']) and not FORCE_RETRAIN:
        print("[TransE] 模型已存在，跳过训练")
        ckpt = torch.load(TRAINED_MODEL_PATHS['TransE'], map_location='cpu')
        return ckpt['model_state_dict']['E.weight'].to(device)

    print("🚀 开始训练 TransE...")
    model = TransE(mapper.entity_count, mapper.relation_count, EMBEDDING_DIM).to(device)
    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    model.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"TransE Epoch {epoch+1}")
        for h_list, r_list, t_list in pbar:
            h = torch.tensor([mapper.entity_to_id[h] for h in h_list], device=device)
            r = torch.tensor([mapper.relation_to_id[r] for r in r_list], device=device)
            t = torch.tensor([mapper.entity_to_id[t] for t in t_list], device=device)
            neg_t = torch.randint(0, mapper.entity_count, (len(h), NEGATIVE_SAMPLES), device=device)
            pos_score = model(h, r, t)
            neg_score = model(h.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              r.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              neg_t.reshape(-1)).reshape(-1, NEGATIVE_SAMPLES)
            loss = torch.mean(torch.relu(pos_score.unsqueeze(1) - neg_score + 1.0))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            model.normalize_entities()
            epoch_loss += loss.item(); pbar.set_postfix(loss=loss.item())
        print(f"[TransE] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'entity_count': mapper.entity_count,
        'relation_count': mapper.relation_count,
        'embedding_dim': EMBEDDING_DIM,
        'entity_to_id': mapper.entity_to_id,
        'relation_to_id': mapper.relation_to_id,
    }, TRAINED_MODEL_PATHS['TransE'])
    print("[TransE] 模型已保存")
    return model.E.weight.data.detach()

# ==================== Train GraphSAGE ====================
# 在函数外部或顶部定义
def negative_sampling_simple(pos_edge_index, num_neg_samples, num_nodes, device):
    """简单负采样：仅替换尾实体（t -> t'）"""
    neg_tail_index = torch.randint(0, num_nodes, (num_neg_samples,), device=device)
    return torch.stack([
        pos_edge_index[0],  # 头实体不变
        neg_tail_index
    ], dim=0)
    
def train_GraphSAGE(triples, mapper, device):
    
    if os.path.exists(TRAINED_MODEL_PATHS['GraphSAGE']) and not FORCE_RETRAIN:
        print("[GraphSAGE] 模型已存在，跳过训练")
        ckpt = torch.load(TRAINED_MODEL_PATHS['GraphSAGE'], map_location=device)
        return ckpt['E.weight']

    print("🚀 开始训练 GraphSAGE...")
    edge_list = []
    for h, r, t in triples:
        if h in mapper.entity_to_id and t in mapper.entity_to_id:
            h_id, t_id = mapper.entity_to_id[h], mapper.entity_to_id[t]
            edge_list.append([h_id, t_id])
            edge_list.append([t_id, h_id])  # 无向

    if len(edge_list) == 0:
        raise ValueError("训练数据为空或实体未正确映射")

    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous().to(device)
    data = Data(edge_index=edge_index, num_nodes=mapper.entity_count)
    model = GraphSAGE(mapper.entity_count, EMBEDDING_DIM).to(device)
    model.edge_index = edge_index  # 用于推理
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0
        for _ in range(50):  # 模拟多轮采样
            z = model.get_embedding_matrix(device)
            pos_edge_index = data.edge_index
            neg_edge_index = custom_negative_sampling(pos_edge_index, num_nodes=data.num_nodes)

            pos_score = (z[pos_edge_index[0]] * z[pos_edge_index[1]]).sum(dim=1)
            neg_score = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)

            loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean() \
                   - torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        print(f"GraphSAGE Epoch {epoch+1} Loss: {epoch_loss / 50:.4f}")

    final_E = model.embedding.weight.data.detach()
    torch.save({
        'model_state_dict': model.state_dict(),
        'E.weight': final_E,
        'entity_count': mapper.entity_count,
        'embedding_dim': EMBEDDING_DIM,
        'entity_to_id': mapper.entity_to_id,
    }, TRAINED_MODEL_PATHS['GraphSAGE'])
    print("[GraphSAGE] 模型已保存")
    return final_E
    
    # if os.path.exists(TRAINED_MODEL_PATHS['GraphSAGE']) and not FORCE_RETRAIN:
    #     print("[GraphSAGE] 模型已存在，跳过训练")
    #     ckpt = torch.load(TRAINED_MODEL_PATHS['GraphSAGE'], map_location='cpu')
    #     return ckpt['E.weight'].to(device)

    # print("🚀 开始训练 GraphSAGE (GraphSAINT-RW)...")

    # # 构建无向边列表
    # edge_list = []
    # for h, r, t in triples:
    #     if h in mapper.entity_to_id and t in mapper.entity_to_id:
    #         h_id = mapper.entity_to_id[h]
    #         t_id = mapper.entity_to_id[t]
    #         edge_list.append([h_id, t_id])
    #         edge_list.append([t_id, h_id])  # 无向图

    # if len(edge_list) == 0:
    #     raise ValueError("No valid triples to build graph.")

    # edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    # data = Data(edge_index=edge_index, num_nodes=mapper.entity_count)

    # # ✅ 使用 GraphSAINTRandomWalkSampler（支持 walk_length）
    # loader = GraphSAINTRandomWalkSampler(
    #     data,
    #     batch_size=128,
    #     walk_length=2,         # ✅ 支持！用于生成游走路径
    #     num_steps=5,           # 每个 epoch 的步数
    #     sample_coverage=100,   # 可选：控制采样覆盖
    #     num_workers=0
    # )

    # model = GraphSAGE(mapper.entity_count, EMBEDDING_DIM).to(device)
    # optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    # model.train()

    # for epoch in range(EPOCHS):
    #     epoch_loss = 0
    #     pbar = tqdm(loader, desc=f"GraphSAGE Epoch {epoch+1}")
    #     for batch in pbar:
    #         batch = batch.to(device)
    #         z = model(torch.arange(batch.num_nodes, device=device), batch.edge_index)

    #         # 正样本得分
    #         pos_score = (z[batch.edge_index[0]] * z[batch.edge_index[1]]).sum(dim=1)
    #         neg_edge_index = negative_sampling_simple(
    #             batch.edge_index,
    #             num_neg_samples=batch.edge_index.size(1),
    #             num_nodes=batch.num_nodes,
    #             device=device
    #         )
    #         neg_score = (z[neg_edge_index[0]] * z[neg_edge_index[1]]).sum(dim=1)

    #         # 损失函数
    #         loss = -torch.log(torch.sigmoid(pos_score) + 1e-8).mean() \
    #                - torch.log(1 - torch.sigmoid(neg_score) + 1e-8).mean()

    #         optimizer.zero_grad()
    #         loss.backward()
    #         optimizer.step()

    #         epoch_loss += loss.item()
    #         pbar.set_postfix(loss=loss.item())

    #     print(f"[GraphSAGE] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    # # 推理：全图嵌入
    # model.eval()
    # with torch.no_grad():
    #     full_x = torch.arange(mapper.entity_count, device=device)
    #     full_edge_index = edge_index.to(device)
    #     final_z = model(full_x, full_edge_index).detach()

    # # 保存
    # torch.save({
    #     'model_state_dict': model.state_dict(),
    #     'E.weight': final_z.cpu(),
    #     'entity_count': mapper.entity_count,
    #     'embedding_dim': EMBEDDING_DIM,
    #     'entity_to_id': mapper.entity_to_id,
    # }, TRAINED_MODEL_PATHS['GraphSAGE'])

    # print("[GraphSAGE] 模型已保存")
    # return final_z

# ==================== Train TransH with Concat Init ====================
def train_TransH_with_concat(train_dataset, mapper, device, e1_weight, e2_weight):
    assert e1_weight.shape[1] == e2_weight.shape[1] == EMBEDDING_DIM
    fused_dim = EMBEDDING_DIM * 2
    print(f"🔧 使用拼接初始化 TransH，输入维度: {fused_dim}")

    model = TransH(mapper.entity_count, mapper.relation_count, fused_dim).to(device)
    combined_E = torch.cat([e1_weight, e2_weight], dim=1)  # [N, 200]
    model.E.weight.data.copy_(combined_E)

    loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    model.train()

    for epoch in range(EPOCHS):
        epoch_loss = 0
        pbar = tqdm(loader, desc=f"TransH Epoch {epoch+1}")
        for h_list, r_list, t_list in pbar:
            h = torch.tensor([mapper.entity_to_id[h] for h in h_list], device=device)
            r = torch.tensor([mapper.relation_to_id[r] for r in r_list], device=device)
            t = torch.tensor([mapper.entity_to_id[t] for t in t_list], device=device)
            neg_t = torch.randint(0, mapper.entity_count, (len(h), NEGATIVE_SAMPLES), device=device)
            pos_score = model(h, r, t)
            neg_score = model(h.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              r.unsqueeze(1).expand(-1, NEGATIVE_SAMPLES).reshape(-1),
                              neg_t.reshape(-1)).reshape(-1, NEGATIVE_SAMPLES)
            loss = torch.mean(torch.relu(pos_score.unsqueeze(1) - neg_score + 1.0))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            model.normalize_entities()
            epoch_loss += loss.item(); pbar.set_postfix(loss=loss.item())
        print(f"[TransH] Epoch {epoch+1} Loss: {epoch_loss / len(loader):.4f}")

    torch.save({
        'model_state_dict': model.state_dict(),
        'entity_count': mapper.entity_count,
        'relation_count': mapper.relation_count,
        'embedding_dim': fused_dim,
        'entity_to_id': mapper.entity_to_id,
        'relation_to_id': mapper.relation_to_id,
    }, TRAINED_MODEL_PATHS['TransH'])
    print("[TransH] 模型已保存")
    return model

# ==================== Evaluate & Predict ====================
def evaluate_and_predict(model, dev_data, test_data, mapper, device):
    entity_emb = model.E.weight.data.cpu().numpy()
    index = faiss.IndexFlatL2(entity_emb.shape[1])
    index.add(entity_emb)

    # Dev 评估
    hits_at = {1:0, 3:0, 10:0}; mrr = 0; count = 0
    for h, r, t in tqdm(dev_data.triples, desc="Evaluating"):
        try:
            h_id = torch.tensor([mapper.entity_to_id[h]], device=device)
            r_id = torch.tensor([mapper.relation_to_id[r]], device=device)
            t_id = mapper.entity_to_id[t]
        except KeyError: continue
        query = model.get_query_embedding(h_id, r_id).cpu().detach().numpy()
        _, indices = index.search(query, 1000)
        pred_ids = indices[0]
        rank = np.where(pred_ids == t_id)[0]
        final_rank = rank[0] + 1 if len(rank) > 0 else 10000
        for k in hits_at: hits_at[k] += 1 if final_rank <= k else 0
        mrr += 1.0 / final_rank; count += 1

    for k in hits_at: hits_at[k] /= count
    mrr /= count
    print(f"HITS@1: {hits_at[1]:.4f}, HITS@3: {hits_at[3]:.4f}, HITS@10: {hits_at[10]:.4f}, MRR: {mrr:.4f}")

    # Test 预测
    results = []
    for h, r, _ in tqdm(test_data.triples, desc="Predict"):
        try:
            h_id = torch.tensor([mapper.entity_to_id[h]], device=device)
            r_id = torch.tensor([mapper.relation_to_id[r]], device=device)
        except KeyError:
            preds = [h] * 10
            results.append('\t'.join([h, r] + preds))
            continue
        q = model.get_query_embedding(h_id, r_id).cpu().detach().numpy()
        _, indices = index.search(q, 10)
        preds = [mapper.id_to_entity[i] for i in indices[0]]
        results.append('\t'.join([h, r] + preds))

    os.makedirs(os.path.dirname(OUTPUT_FILE_PATH), exist_ok=True)
    with open(OUTPUT_FILE_PATH, 'w', encoding='utf-8') as f:
        f.write('\n'.join(results) + '\n')

    zip_path = OUTPUT_FILE_PATH.replace(".tsv", "") + f"__{scheme_type}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        zf.write(OUTPUT_FILE_PATH, arcname=os.path.basename(OUTPUT_FILE_PATH))
        
    print(f"✅ 预测结果已保存: {OUTPUT_FILE_PATH}")

# ==================== 主函数 ====================
def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if hasattr(torch, 'mps') and torch.backends.mps.is_available() else 'cpu')
    print(f"🚀 使用设备: {device}")

    train_data = KnowledgeGraphDataset(TRAIN_FILE_PATH, max_lines=MAX_LINES, is_train=True)
    dev_data = KnowledgeGraphDataset(DEV_FILE_PATH, is_test=False, is_train=False)
    test_data = KnowledgeGraphDataset(TEST_FILE_PATH, is_test=True, is_train=False)
    mapper = EntityRelationMapper()
    mapper.build_mappings(train_data, dev_data, test_data)
    print(f"实体数: {mapper.entity_count}, 关系数: {mapper.relation_count}")

    # 支路2: GraphSAGE
    graphsage_E = train_GraphSAGE(train_data.triples, mapper, device)
    # 支路1: TransE
    transE_E = train_TransE(train_data, mapper, device)
    # TransH 暖启动
    transH_model = train_TransH_with_concat(train_data, mapper, device, transE_E, graphsage_E)
    # 评估与预测
    evaluate_and_predict(transH_model, dev_data, test_data, mapper, device)

if __name__ == "__main__":
    main()